<a href="https://colab.research.google.com/github/AhnafTouseef/Blender-Colab-render-pipline/blob/main/Rendering_pipeline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title # ***`File navigation commands`***
try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

import requests

# Fetch the raw text of whatever script you need
script_url = "https://raw.githubusercontent.com/AhnafTouseef/colab-file-navigator/main/colab_tools.py"
remote_code = requests.get(script_url).text

# Run it immediately into the cell's memory
exec(remote_code)

Mounted at /content/drive
Type "help()" for help


# ***`Blender Installation`***
---

In [2]:
# @title
import subprocess
from pathlib import Path
import re
import requests
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# Configuration
# --------------------------------------------------

path = Path("/content")
BLENDER_RELEASE_URL = "https://download.blender.org/release/"

# --------------------------------------------------
# Check existing Blender installation
# --------------------------------------------------

comp = re.compile(r"blender", re.IGNORECASE)

check = []
for item in path.iterdir():
    check.append(bool(comp.search(str(item))))

if True in check:

    print("Blender already installed")

else:

    # --------------------------------------------------
    # Get release branches
    # --------------------------------------------------

    response = requests.get(BLENDER_RELEASE_URL)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    branches = []

    for link in soup.find_all("a"):
        name = link.get("href", "").strip("/")

        if re.fullmatch(r"Blender\d+\.\d+", name):
            branches.append(name)

    branches.sort(
        key=lambda x: tuple(
            map(int, x.replace("Blender", "").split("."))
        ),
        reverse=True
    )

    # --------------------------------------------------
    # Branch dropdown
    # --------------------------------------------------

    branch_dropdown = widgets.Dropdown(
        options=branches,
        value=branches[0],
        description="Release:",
        style={"description_width": "initial"}
    )

    # --------------------------------------------------
    # Version dropdown
    # --------------------------------------------------

    version_dropdown = widgets.Dropdown(
        description="Version:",
        style={"description_width": "initial"}
    )

    # --------------------------------------------------
    # Update available versions
    # --------------------------------------------------

    def update_versions(change=None):

        branch = branch_dropdown.value

        url = f"{BLENDER_RELEASE_URL}{branch}/"

        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        versions = []

        pattern = re.compile(
            rf"blender-({re.escape(branch.replace('Blender', ''))}"
            r"\.\d+)-linux-x64\.tar\.xz$"
        )

        for link in soup.find_all("a"):

            filename = link.get("href", "")

            match = pattern.fullmatch(filename)

            if match:
                versions.append(match.group(1))

        versions.sort(
            key=lambda x: tuple(map(int, x.split("."))),
            reverse=True
        )

        version_dropdown.options = versions

        if versions:
            version_dropdown.value = versions[0]


    branch_dropdown.observe(update_versions, names="value")

    # --------------------------------------------------
    # Download button
    # --------------------------------------------------

    download_button = widgets.Button(
        description="Download Blender",
        button_style="success"
    )

    output = widgets.Output()

    def download_blender(button):

        with output:

            clear_output()

            version = version_dropdown.value

            filename = f"blender-{version}-linux-x64.tar.xz"

            branch = branch_dropdown.value

            url = f"{BLENDER_RELEASE_URL}{branch}/{filename}"

            archive = path / filename
            install_dir = path / f"blender-{version}-linux-x64"

            print(f"Version: {version}")
            print(f"URL: {url}")
            print()
            print("Downloading...")

            subprocess.run(
                ["wget", "-q", "--show-progress", url, "-O", str(archive)],
                check=True
            )

            print()
            print("Extracting...")

            subprocess.run(
                ["tar", "xf", str(archive), "-C", str(path)],
                check=True
            )

            delete(str(archive))

            print()
            print(f"Blender installed: {install_dir}")


    download_button.on_click(download_blender)

    # --------------------------------------------------
    # Display UI
    # --------------------------------------------------

    display(
        widgets.VBox([
            branch_dropdown,
            version_dropdown,
            download_button,
            output
        ])
    )

    # Load initial versions
    update_versions()

# ***`Configs`***
---

In [ ]:
#@markdown ## 🏢 Step 1: Core Worker & Project Setup


TOTAL_WORKER = 5 # @param {type:"integer"}
RENDER_FILE = "barbershop_interior" # @param {type:"string"}

In [21]:
import os

#@markdown ---
#@markdown ## ⚙️ Step 2: Render Engine & Performance

RENDER_ENGINE = "CYCLES" # @param ["CYCLES","BLENDER_EEVEE_NEXT","BLENDER_WORKBENCH"] {type:"string"}
RENDER_SAMPLES = 128 # @param {type:"integer"}
RESOLUTION_PERCENT = 100 # @param {type:"slider", min:0, max:100, step:1}


#@markdown ### 🎯 Adaptive Sampling Settings

ADAPTIVE_SAMPLING = True # @param {type:"boolean"}
ADAPTIVE_THRESHOLD = 0.1 # @param {type:"number"}


#@markdown ---
#@markdown ---
#@markdown ## 🧼 Step 3: Denoising & Post-Processing

USE_DENOISE = True # @param {type:"boolean"}
DENOISER = "OPENIMAGEDENOISE" # @param ["OPTIX", "OPENIMAGEDENOISE", "NONE"] {type:"string"}


#@markdown ---
#@markdown ---
#@markdown ## 🎬 Step 4: Output & Animation Settings

RENDER_ANIMATION = False # @param {type:"boolean"}
FPS = 24 # @param {type:"integer"}
OUTPUT_FORMAT = "PNG" # @param ["PNG", "JPEG", "OPEN_EXR", "FFMPEG"] {type:"string"}
USE_PERSISTENT_DATA = True # @param {type:"boolean"}
SPATIAL_SPLITS = True # @param {type:"boolean"}


#@markdown ---
#@markdown ---
#@markdown ## 📤 Step 5: Settings to Pass to Blender

PASS_RENDER_ENGINE = False # @param {type:"boolean"}
PASS_RENDER_SAMPLES = False # @param {type:"boolean"}
PASS_RESOLUTION_PERCENT = False # @param {type:"boolean"}

PASS_ADAPTIVE_SAMPLING = False # @param {type:"boolean"}
PASS_ADAPTIVE_THRESHOLD = False # @param {type:"boolean"}

PASS_USE_DENOISE = False # @param {type:"boolean"}
PASS_DENOISER = False # @param {type:"boolean"}

PASS_RENDER_ANIMATION = False # @param {type:"boolean"}
PASS_FPS = False # @param {type:"boolean"}
PASS_OUTPUT_FORMAT = False # @param {type:"boolean"}
PASS_USE_PERSISTENT_DATA = False # @param {type:"boolean"}
PASS_SPATIAL_SPLITS = False # @param {type:"boolean"}


#@markdown ---
#@markdown ### 🧩 Advanced Blender Python

MACRO = "" # @param {type:"string"}
PASS_MACRO = False # @param {type:"boolean"}


# --------------------------------------------------
# Configuration directory
# --------------------------------------------------

CONFIG_DIR = "/content/drive/MyDrive/Render_Farm"

# Ensure the directory exists
os.makedirs(CONFIG_DIR, exist_ok=True)


# --------------------------------------------------
# Build Blender settings
# --------------------------------------------------

settings = []


if PASS_RENDER_ENGINE:
    settings.append(
        f"S.render.engine = '{RENDER_ENGINE}'"
    )


if PASS_RENDER_SAMPLES:
    settings.append(
        f"S.cycles.samples = {RENDER_SAMPLES}"
    )


if PASS_RESOLUTION_PERCENT:
    settings.append(
        f"S.render.resolution_percentage = {RESOLUTION_PERCENT}"
    )


if PASS_ADAPTIVE_SAMPLING:
    settings.append(
        f"S.cycles.use_adaptive_sampling = {ADAPTIVE_SAMPLING}"
    )


if PASS_ADAPTIVE_THRESHOLD:
    settings.append(
        f"S.cycles.adaptive_threshold = {ADAPTIVE_THRESHOLD}"
    )


if PASS_USE_DENOISE:
    settings.append(
        f"S.cycles.use_denoising = {USE_DENOISE}"
    )


if PASS_DENOISER:
    settings.append(
        f"S.cycles.denoiser = '{DENOISER}'"
    )


if PASS_RENDER_ANIMATION:
    settings.append(
        f"RENDER_ANIMATION = {RENDER_ANIMATION}"
    )


if PASS_FPS:
    settings.append(
        f"S.render.fps = {FPS}"
    )


if PASS_OUTPUT_FORMAT:
    settings.append(
        f"S.render.image_settings.file_format = '{OUTPUT_FORMAT}'"
    )


if PASS_USE_PERSISTENT_DATA:
    settings.append(
        f"S.render.use_persistent_data = {USE_PERSISTENT_DATA}"
    )


if PASS_SPATIAL_SPLITS:
    settings.append(
        f"S.cycles.debug_use_spatial_splits = {SPATIAL_SPLITS}"
    )


if PASS_MACRO and MACRO.strip():
    settings.append(MACRO)


# --------------------------------------------------
# Create render_settings.py
# --------------------------------------------------

render_settings_content = """import bpy

S = bpy.context.scene

""" + "\n".join(settings) + "\n"


render_settings_file_path = os.path.join(
    CONFIG_DIR,
    "render_settings.py"
)


with open(render_settings_file_path, "w") as f:
    f.write(render_settings_content)


# --------------------------------------------------
# Create config.json
# --------------------------------------------------

config_file_path = os.path.join(
    CONFIG_DIR,
    "config.json"
)


with open(config_file_path, "w") as f:
    f.write(
        '{\n'
        f'"total_workers" : {TOTAL_WORKER},\n'
        f'"blend_file" : "{RENDER_FILE}"\n'
        '}'
    )


# --------------------------------------------------
# Result
# --------------------------------------------------

print(f"Blender render settings saved to:")
print(render_settings_file_path)

print(f"\nSettings passed: {len(settings)}")

Blender render settings saved to /content/drive/MyDrive/Render_Farm/render_settings.py


# ***`Render Setup`***
---

In [3]:
# @title
import os
import sys
import json
import re
import subprocess
from IPython.display import clear_output

# ============================================================
# SETTINGS
# ============================================================

MAIN_DIRECTORY = "/content/drive/MyDrive/Render_Farm"

CONFIG_FILE = f"{MAIN_DIRECTORY}/config.json"
PYTHON_OVERRIDE = f"{MAIN_DIRECTORY}/render_settings.py"
REGISTER_DIR = f"{MAIN_DIRECTORY}/register"
OUTPUT_DIR = f"{MAIN_DIRECTORY}/output"

BLENDER_EXEC = f"/content/blender-{version_dropdown.value}-linux-x64/blender"

os.makedirs(REGISTER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

TOTAL_WORKERS = int(config["total_workers"])
BLEND_PATH = f"{MAIN_DIRECTORY}/{config['blend_file']}"


# ============================================================
# WORKER
# ============================================================

def start_render():
    def register_worker():
        ids = [int(os.path.splitext(x)[0]) for x in os.listdir(REGISTER_DIR) if os.path.splitext(x)[0].isdigit()]
        worker = max(ids) + 1 if ids else 1

        if worker == TOTAL_WORKERS:
            print(f"{worker}th worker arived. Cleaning registry")
            for file in os.listdir(REGISTER_DIR):
                os.remove(f"{REGISTER_DIR}/{file}")
            return worker
        else:
            open(f"{REGISTER_DIR}/{worker}.txt", "w").close()
            print(f"Worker {worker} registered")
            return worker


    worker_id = register_worker()


    # ============================================================
    # BLENDER DATA
    # ============================================================

    def get_scene_data():
        code = "import bpy; c=bpy.context.scene; print(f'START:{c.frame_start}\\nEND:{c.frame_end}')"

        res = subprocess.run(
            f'{BLENDER_EXEC} -b {BLEND_PATH} --python-expr "{code}"',
            shell=True,
            text=True,
            capture_output=True,
        )

        # Extract numbers directly using split
        print(res.stdout)
        start = res.stdout.split("START:")[1].split("\n")[0]
        end = res.stdout.split("END:")[1].split("\n")[0]
        return {"start_frame":start, "end_frame":end}


    scene = get_scene_data()

    START_FRAME = int(scene["start_frame"])
    END_FRAME = int(scene["end_frame"])



    def split_frames(worker_index, total_workers, start_frame, end_frame):
        total_frame = end_frame - start_frame + 1
        division, remainder = divmod(total_frame, total_workers)
        size = division + (1 if worker_index < remainder else 0)
        offset = worker_index * division + min(worker_index, remainder)
        return start_frame + offset, start_frame + offset + size - 1


    my_start, my_end = split_frames(worker_id - 1, TOTAL_WORKERS, START_FRAME, END_FRAME)



    # ============================================================
    # BLENDER COMMAND
    # ============================================================


    cmd = [BLENDER_EXEC, "-b", BLEND_PATH,"-P", PYTHON_OVERRIDE,"-o", f"{OUTPUT_DIR}/Frame_","-s", str(my_start), "-e", str(my_end), "-a","--", "--cycles-device", "OPTIX"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    finder = re.compile(r"Fra:(\d+)")


    # ============================================================
    # LIVE MONITOR
    # ============================================================

    current_frame = my_start

    for line in process.stdout:
        # clear_output(wait=True)
        # print(line)

        try:
          frame = finder.search(line).group(1)
        except:
          continue
        # print(frame)
        if frame:
            current_frame = int(frame)

        frame_ratio = (current_frame - my_start + 1) / (my_end - my_start + 1)

        frame_bar = "█" * int(frame_ratio * 30) + "-" * (30 - int(frame_ratio * 30))

        clear_output(wait=True)
        print(f"Project name: {config['blend_file']}, Start Frame: {START_FRAME}, End Frame{END_FRAME}")

        print("=" * 50)
        print(f"Worker {worker_id}/{TOTAL_WORKERS}")
        print(f"Frames {my_start} -> {my_end}")
        print(f"Output {OUTPUT_DIR}")
        print("=" * 50)

        print("=" * 50)
        print("Blender Render Monitor")
        print("=" * 50)
        print(f"Frame   : {current_frame}/{my_end}")
        print(f"Overall : [{frame_bar}] {frame_ratio * 100:.2f}%")




    process.wait()


    if process.returncode == 0:
        print("\n✅ Render finished")
    else:
        print("\n❌ Render failed")

# ***`Monitor`***
---

In [4]:
start_render()

Worker 1 registered
00:04.815  blend            | Read blend: "/content/drive/MyDrive/Render_Farm/barbershop_interior.blend"
START:1
END:5
Blender 5.2.2 LTS (hash d13f752e3b9c built 2026-09-15 01:34:58)
scripts disabled for "/content/drive/MyDrive/Render_Farm/barbershop_interior.blend", skipping 'generate_customprops.py'

Blender quit


✅ Render finished
